# An overview of the Spark SQL and Dataframe API

### Attention!

This notebook contains some linux-specific commands to download datasets. If you are running on windows, it is advised to run this notebook through Google Colab.

#### Installing PySpark

In [1]:
# This installs pyspark in the current python environment.
# By installing pyspark, we automatically also install spark.
# You **need** to run this cell when running this notebook in google colab
# Ignore this cell if you already have spark installed.
!pip install pyspark

#### General imports and starting Spark

In [2]:
#This is needed to start a Spark session from the notebook
#You may adjust the memory used by the driver program based on your machine's settings
import os 
os.environ['PYSPARK_SUBMIT_ARGS'] ="--conf spark.driver.memory=3g  pyspark-shell"

from pyspark.sql import SparkSession

In [3]:
import os
import sys
os.chdir(r"C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment")
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
# -------------------------------
# Start Spark in LOCAL mode
# -------------------------------

#The following lines are just there to allow this cell to be re-executed multiple times:
#if a spark session was already started, we stop it before starting a new one
#(there can be only one spark context per jupyter notebook)
try: 
    spark
    print("Spark application already started. Terminating existing application and starting new one")
    spark.stop()
except: 
    pass

# Create a new spark session (note, the * indicates to use all available CPU cores)
spark = SparkSession \
    .builder \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .appName("demoRDD") \
    .getOrCreate()
    
#When dealing with RDDs, we work the sparkContext object. See https://spark.apache.org/docs/latest/api/python/pyspark.html#pyspark.SparkContext
sc=spark.sparkContext

# We print the sparkcontext. This prints general information about the spark instance we have connected to. 
# In particular, the hyperlink allows us to open the spark UI (useful for seeing what is going on)
# Note: this hyperlink won't work when running this notebook in Google Colab.
sc

C:\Users\gihame\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


<SparkContext master=local[*] appName=demoRDD>

### Downloading data

The next cell downloads the data we will work on.

In [4]:
# Local data is already in this folder. wget/unzip/ls are Linux-only.
import os
root = r"C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment"
os.chdir(root)
print("Working directory:", os.getcwd())
for name in ["books", "sensors", "movielens", "github"]:
    folder = os.path.join("data", name)
    print(f"{folder}: {len(os.listdir(folder))} files")

Working directory: C:\Users\gihame\Documents\UHasselt\First Year\Advanced Topic in DS\spark-exercise-assignment
data\books: 30 files
data\sensors: 1 files
data\movielens: 5 files
data\github: 2 files


### Loading a file as a dataframe
The file “data/sensors/sensor-sample.txt” is a text file where each line consist of multiple fields of information, let’s call them : Date(Date), Time(Time), RoomId(Integer)-SensorId(Integer), Value1(float), Value2(float)

In [5]:
with open("data/sensors/sensor-sample.txt", encoding="utf-8") as f:
    for i, line in zip(range(5), f):
        print(line, end="")

2017-03-31 03:38:16.508 1-0 122.153 2.03397
2017-03-31 03:38:15.967 1-1 -3.91901 2.09397
2017-03-31 03:38:16.577 1-2 11.04 2.07397
2017-02-28 00:59:16.359 1-0 19.9884 2.74964
2017-02-28 00:59:16.803 1-1 37.0933 2.76964


In [6]:
# Try to load the sensor datafile as a Spark Dataframe by means of the read.csv option
sensorDF = spark.read.csv("data/sensors/sensor-sample.txt")
# Dataframes have a show() action that by defaultretrieves the first 20 rows (pass an argument to get more/less)
sensorDF.show(5)

+--------------------+
|                 _c0|
+--------------------+
|2017-03-31 03:38:...|
|2017-03-31 03:38:...|
|2017-03-31 03:38:...|
|2017-02-28 00:59:...|
|2017-02-28 00:59:...|
+--------------------+
only showing top 5 rows


In [7]:
# The above did not work because CSV expects columns to be delimited by commans, not spaces
# However, we can tell spark to use spaces as delimiters instead
sensorDF = spark.read.option("delimiter", " ").csv("data/sensors/sensor-sample.txt")
sensorDF.show(5)

+----------+------------+---+--------+-------+
|       _c0|         _c1|_c2|     _c3|    _c4|
+----------+------------+---+--------+-------+
|2017-03-31|03:38:16.508|1-0| 122.153|2.03397|
|2017-03-31|03:38:15.967|1-1|-3.91901|2.09397|
|2017-03-31|03:38:16.577|1-2|   11.04|2.07397|
|2017-02-28|00:59:16.359|1-0| 19.9884|2.74964|
|2017-02-28|00:59:16.803|1-1| 37.0933|2.76964|
+----------+------------+---+--------+-------+
only showing top 5 rows


In [8]:
# By default, no type inference is done when reading a CSV file
# Spark made up column names (since we did not provide any), and gives every column the String type
sensorDF.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)



In [9]:
# The dtypes member is another way to get the types
sensorDF.dtypes

[('_c0', 'string'),
 ('_c1', 'string'),
 ('_c2', 'string'),
 ('_c3', 'string'),
 ('_c4', 'string')]

In [10]:
# We force Spark to treat each column as having the desired type by providing the schema
schema = "date DATE, time TIMESTAMP, room_sensor STRING, Value1 FLOAT, Value2 FLOAT"
sensorDF = spark.read.option("delimiter", " ").csv("data/sensors/sensor-sample.txt", schema=schema)
sensorDF.show(5)
sensorDF.printSchema()

+----------+--------------------+-----------+--------+-------+
|      date|                time|room_sensor|  Value1| Value2|
+----------+--------------------+-----------+--------+-------+
|2017-03-31|2026-10-02 03:38:...|        1-0| 122.153|2.03397|
|2017-03-31|2026-10-02 03:38:...|        1-1|-3.91901|2.09397|
|2017-03-31|2026-10-02 03:38:...|        1-2|   11.04|2.07397|
|2017-02-28|2026-10-02 00:59:...|        1-0| 19.9884|2.74964|
|2017-02-28|2026-10-02 00:59:...|        1-1| 37.0933|2.76964|
+----------+--------------------+-----------+--------+-------+
only showing top 5 rows
root
 |-- date: date (nullable = true)
 |-- time: timestamp (nullable = true)
 |-- room_sensor: string (nullable = true)
 |-- Value1: float (nullable = true)
 |-- Value2: float (nullable = true)



In [11]:
# We can query the dataframe using SQL by first registering the dataframe as a "table", 
# and then executing a SQL statement
sensorDF.createOrReplaceTempView('sensors')
# The result of calling spark.sql(...) is again a dataframe
spark.sql("select distinct(date) from sensors").show()

+----------+
|      date|
+----------+
|2017-02-28|
|2017-03-13|
|2017-03-05|
|2017-03-21|
|2017-03-24|
|2017-03-11|
|2017-03-06|
|2017-03-26|
|2017-03-07|
|2017-03-27|
|2017-03-10|
|2017-03-03|
|2017-03-23|
|2017-03-22|
|2017-03-02|
|2017-03-12|
|2017-03-31|
|2017-03-09|
|2017-03-04|
|2017-03-01|
+----------+
only showing top 20 rows


In [12]:
# we can alternatively use methods to encode the SQL statement
sensorDF.select("date").show()

+----------+
|      date|
+----------+
|2017-03-31|
|2017-03-31|
|2017-03-31|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
|2017-02-28|
+----------+
only showing top 20 rows


In [13]:
# a dataframe is an RDD, so all RDD methods still work
sensorDF.select("date").distinct().show()

+----------+
|      date|
+----------+
|2017-02-28|
|2017-03-13|
|2017-03-05|
|2017-03-21|
|2017-03-24|
|2017-03-11|
|2017-03-06|
|2017-03-26|
|2017-03-07|
|2017-03-27|
|2017-03-10|
|2017-03-03|
|2017-03-23|
|2017-03-22|
|2017-03-02|
|2017-03-12|
|2017-03-31|
|2017-03-09|
|2017-03-04|
|2017-03-01|
+----------+
only showing top 20 rows


## Loading other file formats

The file `data/github/2015-03-01-0.json.gz` is a compressed file conainting multiple JSON objects, one per line. Here is a sample line of this file, neatly formatted:
`{ "id": "2614896652",
    "type": "CreateEvent",
    "actor": {
        "id": 739622,
        "login": "treydock",
        "gravatar_id": "",
        "url": "https://api.githb.com/users/treydock",
        "avatar_url": "https://avatars.githubusercontent.com/u/739622?"
    },
    "repo": {
        "id": 23934080,
        "name": "Early-Modern-OCR/emop-dashboard",
    "url": "https://api.github.com/repos/Early-Modern-OCR/emop-dashboard"
    },
    "payload": {
        "ref": "development",
        "ref_type": "branch",
        "master-branch": "master",
        "description": "",
        "pusher_type": "user",
    },
    "public": true,
    "created_at": "2015-03-01T00:00:00Z",
    "org": {
        "id": 10965476,
        "login": "Early-Modern-OCR",
        "gravatar_id": "",
        "url": "https://api.github.com/orgs/Early-Modern-OCR",
        "avatar_url": "https://avatars.githubusercontent.com/u/10965476?"
    }
}`


In [14]:
# We can load the compressed json file as a dataframe as follows:
gitDF = spark.read.json('data/github/2015-03-01-0.json.gz')

# This has the columns actor, created_at, id, org, ...
gitDF.show(2)

+--------------------+--------------------+----------+--------------------+--------------------+------+--------------------+-----------+
|               actor|          created_at|        id|                 org|             payload|public|                repo|       type|
+--------------------+--------------------+----------+--------------------+--------------------+------+--------------------+-----------+
|{https://avatars....|2015-03-01T00:00:00Z|2614896652|{https://avatars....|{NULL, NULL, NULL...|  true|{23934080, Early-...|CreateEvent|
|{https://avatars....|2015-03-01T00:00:00Z|2614896653|                NULL|{NULL, 6dda286a3a...|  true|{31481156, bezerr...|  PushEvent|
+--------------------+--------------------+----------+--------------------+--------------------+------+--------------------+-----------+
only showing top 2 rows


In [15]:
# But some of these fields are themselves records!
gitDF.printSchema()

root
 |-- actor: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- org: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- payload: struct (nullable = true)
 |    |-- action: string (nullable = true)
 |    |-- before: string (nullable = true)
 |    |-- comment: struct (nullable = true)
 |    |    |-- _links: struct (nullable = true)
 |    |    |    |-- html: struct (nullable = true)
 |    |    |    |    |-- href: string (nullable = true)
 |    |    |    |-- pull_request: struct (nullable = true)
 |    |    |    |    |-- href: strin

In [16]:
# We can access these nested fields in both the API and SQL

# Select the actor (each actor is itself a record)
gitDF.select('actor').show(2)

# select the actor login (i.e., the login field inside the actor field)
gitDF.select('actor.login').show(2)

+--------------------+
|               actor|
+--------------------+
|{https://avatars....|
|{https://avatars....|
+--------------------+
only showing top 2 rows


+----------+
|     login|
+----------+
|  treydock|
|bezerrathm|
+----------+
only showing top 2 rows


In [17]:
gitDF.createOrReplaceTempView('git')
spark.sql("select actor.login from git").show(2)

+----------+
|     login|
+----------+
|  treydock|
|bezerrathm|
+----------+
only showing top 2 rows


In [18]:
sc.stop()